# The Riemann--Stieltjes Integral and Applications

## An interactive companion to the chapter

This notebook develops the central ideas of the chapter through mathematical explanations, visual experiments, and short self-check activities. Its purpose is not merely to produce numerical answers. Each experiment is designed to show **why** a theorem has the form stated in the chapter and **how** the hypotheses affect the result.

The guiding object is

$$
\int_a^b f(x)\,d\alpha(x),
$$

where the increments $\Delta x_i$ of ordinary Riemann integration are replaced by the increments

$$
\Delta\alpha_i=\alpha(x_i)-\alpha(x_{i-1}).
$$

> **How to use the notebook.** Run the cells from top to bottom. Change the controls and press the blue buttons. Every numerical graph should be interpreted together with the mathematical statement immediately above it. Numerical evidence illustrates a theorem; it does not replace its proof.

## Learning map

By the end of the notebook, you should be able to:

1. construct tagged Riemann--Stieltjes sums;
2. compare upper and lower Darboux--Stieltjes sums;
3. verify the reduction $\int f\,d\alpha=\int f\alpha'\,dx$ for smooth integrators;
4. interpret jumps of $\alpha$ as point masses;
5. visualize total variation and Jordan decomposition;
6. study the weighted midpoint rule and its error bound;
7. approximate an indefinite or improper Stieltjes integral; and
8. compute expectations for a mixed probability distribution.

The controls use `ipywidgets`. They work in Jupyter Notebook, JupyterLab, and Google Colab environments in which widgets are enabled.

In [ ]:
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
except ImportError:
    print('ipywidgets is missing; installing it now...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets'])
    import ipywidgets as widgets
from IPython.display import display, HTML, Math, clear_output

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 4.8),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 2.2
})

BLUE = '#2563eb'
ORANGE = '#f97316'
GREEN = '#16a34a'
RED = '#dc2626'
PURPLE = '#7c3aed'

def grid_integral(y, x):
    """Composite trapezoidal integration, compatible with old and new NumPy."""
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))

def cumulative_grid_integral(y, x):
    """Cumulative composite-trapezoidal integral with initial value zero."""
    increments = 0.5 * (y[1:] + y[:-1]) * np.diff(x)
    return np.concatenate(([0.0], np.cumsum(increments)))

def rs_sum(f, alpha, a, b, n, tag='midpoint'):
    """Return a tagged Riemann--Stieltjes sum and its ingredients."""
    edges = np.linspace(a, b, int(n) + 1)
    if tag == 'left':
        tags = edges[:-1]
    elif tag == 'right':
        tags = edges[1:]
    else:
        tags = 0.5 * (edges[:-1] + edges[1:])
    delta_alpha = alpha(edges[1:]) - alpha(edges[:-1])
    contributions = f(tags) * delta_alpha
    return float(np.sum(contributions)), edges, tags, delta_alpha, contributions

FUNCTIONS = {
    'f(x) = x': lambda x: x,
    'f(x) = x²': lambda x: x**2,
    'f(x) = sin(2πx)': lambda x: np.sin(2 * np.pi * x),
    'f(x) = exp(x)': np.exp,
    'f(x) = 1/(1+x²)': lambda x: 1 / (1 + x**2),
}

INTEGRATORS = {
    'α(x) = x': (lambda x: x, lambda x: np.ones_like(x)),
    'α(x) = x²': (lambda x: x**2, lambda x: 2 * x),
    'α(x) = exp(x)': (np.exp, np.exp),
}

display(HTML(
    "<div style='padding:12px 16px;border-left:5px solid #2563eb;"
    "background:#eff6ff;border-radius:6px'><b>Setup complete.</b> "
    "The interactive experiments are ready.</div>"
))

---
## 1. Tagged Riemann--Stieltjes sums

Let

$$P=\{a=x_0<x_1<\cdots<x_n=b\}$$

be a partition and choose a tag $t_i\in[x_{i-1},x_i]$ in each subinterval. The associated sum is

$$
S(P,T;f,\alpha)=\sum_{i=1}^n f(t_i)\bigl[\alpha(x_i)-\alpha(x_{i-1})\bigr].
$$

The factor $f(t_i)$ records the value of the integrand, whereas $\Delta\alpha_i$ is the weight assigned by the integrator. If $\alpha(x)=x$, then $\Delta\alpha_i=\Delta x_i$ and the ordinary Riemann sum is recovered. If $\alpha$ grows rapidly on part of the interval, that region receives greater weight.

The experiment below displays every contribution $f(t_i)\Delta\alpha_i$. Compare left, midpoint, and right tags, and then increase $n$.

In [ ]:
sum_f = widgets.Dropdown(options=list(FUNCTIONS), value='f(x) = x²', description='Integrand:')
sum_alpha = widgets.Dropdown(options=list(INTEGRATORS), value='α(x) = x²', description='Integrator:')
sum_n = widgets.IntSlider(value=8, min=2, max=80, step=1, description='Intervals:', continuous_update=False)
sum_tag = widgets.ToggleButtons(options=['left', 'midpoint', 'right'], value='midpoint', description='Tags:')
sum_button = widgets.Button(description='Compute the sum', button_style='primary', icon='calculator')
sum_reset = widgets.Button(description='Reset', icon='refresh')
sum_output = widgets.Output()

def draw_rs_sum(_=None):
    with sum_output:
        clear_output(wait=True)
        f = FUNCTIONS[sum_f.value]
        alpha, alpha_prime = INTEGRATORS[sum_alpha.value]
        value, edges, tags, delta_alpha, contributions = rs_sum(
            f, alpha, 0.0, 1.0, sum_n.value, sum_tag.value
        )
        fine = np.linspace(0, 1, 20001)
        reference = grid_integral(f(fine) * alpha_prime(fine), fine)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(fine, f(fine), color=BLUE, label='f(x)')
        axes[0].scatter(tags, f(tags), color=ORANGE, zorder=3, label='tag values')
        for edge in edges:
            axes[0].axvline(edge, color='gray', alpha=0.14, linewidth=1)
        axes[0].set_title('Partition and sampled integrand')
        axes[0].set_xlabel('x')
        axes[0].legend()
        
        axes[1].bar(np.arange(1, sum_n.value + 1), contributions, color=GREEN, alpha=0.8)
        axes[1].axhline(0, color='black', linewidth=0.8)
        axes[1].set_title(r'Individual terms $f(t_i)\Delta\alpha_i$')
        axes[1].set_xlabel('subinterval i')
        axes[1].set_ylabel('weighted contribution')
        plt.tight_layout()
        plt.show()
        
        display(HTML(
            f"<h4>Result</h4><p><b>Stieltjes sum:</b> {value:.10f}<br>"
            f"<b>Smooth-reference value:</b> {reference:.10f}<br>"
            f"<b>Absolute error:</b> {abs(value-reference):.3e}</p>"
        ))

def reset_rs_sum(_):
    sum_f.value = 'f(x) = x²'
    sum_alpha.value = 'α(x) = x²'
    sum_n.value = 8
    sum_tag.value = 'midpoint'
    draw_rs_sum()

sum_button.on_click(draw_rs_sum)
sum_reset.on_click(reset_rs_sum)
display(widgets.VBox([
    widgets.HBox([sum_f, sum_alpha]),
    sum_n, sum_tag,
    widgets.HBox([sum_button, sum_reset]),
    sum_output
]))
draw_rs_sum()

---
## 2. Darboux sums and the integrability gap

Assume that $\alpha$ is increasing. On $I_i=[x_{i-1},x_i]$, define

$$m_i=\inf_{x\in I_i}f(x),\qquad M_i=\sup_{x\in I_i}f(x).$$

The lower and upper Stieltjes sums are

$$
L(P,f,\alpha)=\sum_{i=1}^n m_i\Delta\alpha_i,\qquad
U(P,f,\alpha)=\sum_{i=1}^n M_i\Delta\alpha_i.
$$

Their difference is

$$
U-L=\sum_{i=1}^n \operatorname{osc}(f;I_i)\Delta\alpha_i.
$$

Thus the Darboux criterion says that $f\in\mathcal R(\alpha)$ exactly when partitions can be found for which $U-L$ is arbitrarily small. In the computation below, the infima and suprema are estimated on a very fine grid inside each subinterval. This is a numerical illustration, not an exact symbolic proof.

In [ ]:
darboux_f = widgets.Dropdown(options=list(FUNCTIONS), value='f(x) = sin(2πx)', description='Integrand:')
darboux_alpha = widgets.Dropdown(options=list(INTEGRATORS), value='α(x) = x', description='Integrator:')
darboux_n = widgets.IntSlider(value=8, min=2, max=64, step=1, description='Intervals:', continuous_update=False)
darboux_button = widgets.Button(description='Compare L and U', button_style='primary', icon='arrows-alt-h')
darboux_output = widgets.Output()

def approximate_darboux(f, alpha, n, samples_per_interval=301):
    edges = np.linspace(0, 1, n + 1)
    lower = 0.0
    upper = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        local_x = np.linspace(left, right, samples_per_interval)
        local_y = f(local_x)
        weight = float(alpha(np.array([right]))[0] - alpha(np.array([left]))[0])
        lower += float(np.min(local_y)) * weight
        upper += float(np.max(local_y)) * weight
    return lower, upper

def draw_darboux(_=None):
    with darboux_output:
        clear_output(wait=True)
        f = FUNCTIONS[darboux_f.value]
        alpha, alpha_prime = INTEGRATORS[darboux_alpha.value]
        n = darboux_n.value
        lower, upper = approximate_darboux(f, alpha, n)
        ns = np.array(sorted(set([2, 4, 8, 16, 32, 64, n])))
        pairs = np.array([approximate_darboux(f, alpha, int(k)) for k in ns])
        fine = np.linspace(0, 1, 30001)
        reference = grid_integral(f(fine) * alpha_prime(fine), fine)
        
        fig, ax = plt.subplots()
        ax.plot(ns, pairs[:, 0], 'o-', color=BLUE, label='lower sum L')
        ax.plot(ns, pairs[:, 1], 'o-', color=ORANGE, label='upper sum U')
        ax.axhline(reference, color=GREEN, linestyle='--', label='reference integral')
        ax.fill_between(ns, pairs[:, 0], pairs[:, 1], color=PURPLE, alpha=0.12, label='Darboux gap')
        ax.set_xscale('log', base=2)
        ax.set_xlabel('number of subintervals n')
        ax.set_ylabel('value')
        ax.set_title('Refinement squeezes the integral')
        ax.legend()
        plt.show()
        display(HTML(
            f"<p><b>L =</b> {lower:.9f} &nbsp; <b>U =</b> {upper:.9f}<br>"
            f"<b>Gap U − L =</b> {upper-lower:.3e}</p>"
        ))

darboux_button.on_click(draw_darboux)
display(widgets.VBox([widgets.HBox([darboux_f, darboux_alpha]), darboux_n, darboux_button, darboux_output]))
draw_darboux()

---
## 3. Smooth integrators: reduction to an ordinary integral

If $\alpha\in C^1([a,b])$ and $f$ is Riemann integrable, then

$$
\boxed{\displaystyle \int_a^b f(x)\,d\alpha(x)=\int_a^b f(x)\alpha'(x)\,dx.}
$$

The mean value theorem explains the formula: on a small interval,

$$\alpha(x_i)-\alpha(x_{i-1})=\alpha'(\xi_i)(x_i-x_{i-1})$$

for some $\xi_i\in(x_{i-1},x_i)$. Consequently, a Stieltjes sum resembles an ordinary Riemann sum for the product $f\alpha'$.

The next cell compares a finite weighted midpoint sum with a high-resolution approximation of the ordinary integral on the right-hand side.

In [ ]:
smooth_f = widgets.Dropdown(options=list(FUNCTIONS), value='f(x) = exp(x)', description='Integrand:')
smooth_alpha = widgets.Dropdown(options=list(INTEGRATORS), value='α(x) = x²', description='Integrator:')
smooth_n = widgets.IntSlider(value=12, min=2, max=200, step=1, description='Intervals:', continuous_update=False)
smooth_button = widgets.Button(description='Verify reduction', button_style='primary', icon='check')
smooth_output = widgets.Output()

def draw_smooth_reduction(_=None):
    with smooth_output:
        clear_output(wait=True)
        f = FUNCTIONS[smooth_f.value]
        alpha, alpha_prime = INTEGRATORS[smooth_alpha.value]
        approx, *_ = rs_sum(f, alpha, 0, 1, smooth_n.value, 'midpoint')
        x = np.linspace(0, 1, 30001)
        density = f(x) * alpha_prime(x)
        ordinary = grid_integral(density, x)
        
        fig, ax = plt.subplots()
        ax.plot(x, f(x), color=BLUE, label='f(x)')
        ax.plot(x, alpha_prime(x), color=ORANGE, label="α'(x)")
        ax.plot(x, density, color=GREEN, label="f(x)α'(x)")
        ax.fill_between(x, 0, density, color=GREEN, alpha=0.12)
        ax.set_title('The ordinary-integral density generated by α')
        ax.set_xlabel('x')
        ax.legend()
        plt.show()
        display(HTML(
            f"<p><b>Weighted midpoint Stieltjes sum:</b> {approx:.10f}<br>"
            f"<b>Ordinary integral of fα′:</b> {ordinary:.10f}<br>"
            f"<b>Difference:</b> {abs(approx-ordinary):.3e}</p>"
        ))

smooth_button.on_click(draw_smooth_reduction)
display(widgets.VBox([widgets.HBox([smooth_f, smooth_alpha]), smooth_n, smooth_button, smooth_output]))
draw_smooth_reduction()

### Integration by parts

If both Stieltjes integrals exist and $f,\alpha$ have no common discontinuity, then

$$
\int_a^b f\,d\alpha+\int_a^b \alpha\,df=f(b)\alpha(b)-f(a)\alpha(a).
$$

For smooth functions this becomes the familiar product-rule identity because $d\alpha=\alpha' dx$ and $df=f' dx$. The following experiment uses $f(x)=1+x^2$ and $\alpha(x)=e^x$ on $[0,1]$. The two numerical integrals should add to the boundary term.

In [ ]:
ibp_n = widgets.IntSlider(value=20, min=2, max=300, step=1, description='Intervals:', continuous_update=False)
ibp_button = widgets.Button(description='Check integration by parts', button_style='primary', icon='exchange')
ibp_output = widgets.Output()

def draw_ibp(_=None):
    with ibp_output:
        clear_output(wait=True)
        f = lambda x: 1 + x**2
        alpha = np.exp
        left, *_ = rs_sum(f, alpha, 0, 1, ibp_n.value, 'midpoint')
        second, *_ = rs_sum(alpha, f, 0, 1, ibp_n.value, 'midpoint')
        boundary = float(f(np.array([1.0]))[0] * np.e - f(np.array([0.0]))[0])
        residual = left + second - boundary
        labels = [r'$\int f\,d\alpha$', r'$\int \alpha\,df$', 'sum', 'boundary term']
        values = [left, second, left + second, boundary]
        colors = [BLUE, ORANGE, GREEN, PURPLE]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(labels, values, color=colors, alpha=0.82)
        ax.set_title('Numerical integration-by-parts identity')
        ax.set_ylabel('value')
        plt.show()
        display(HTML(
            f"<p><b>Left side:</b> {left+second:.10f}<br>"
            f"<b>Boundary term:</b> {boundary:.10f}<br>"
            f"<b>Residual:</b> {residual:.3e}</p>"
        ))

ibp_button.on_click(draw_ibp)
display(widgets.VBox([ibp_n, ibp_button, ibp_output]))
draw_ibp()

---
## 4. Step integrators and point masses

Let $H_d(x)=0$ for $x<d$ and $H_d(x)=1$ for $x\ge d$. If

$$\alpha(x)=x+cH_d(x),$$

then $\alpha$ contains a continuous part and a jump of size $c$ at $d$. For continuous $f$,

$$
\boxed{\displaystyle \int_a^b f(x)\,d\alpha(x)=\int_a^b f(x)\,dx+c f(d)}.
$$

The jump behaves like a point mass: it assigns the entire weight $c$ to the single value $f(d)$. A uniform numerical partition may locate the jump inside one subinterval. As the mesh tends to zero, the tag in that interval approaches $d$ and the contribution approaches $cf(d)$. A **jump-aware** method treats this contribution explicitly and is therefore much more accurate at a fixed $n$.

In [ ]:
jump_f = widgets.Dropdown(options=list(FUNCTIONS), value='f(x) = exp(x)', description='Integrand:')
jump_size = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Jump c:', continuous_update=False)
jump_point = widgets.FloatSlider(value=0.43, min=0.05, max=0.95, step=0.01, description='Point d:', continuous_update=False)
jump_n = widgets.IntSlider(value=10, min=2, max=150, step=1, description='Intervals:', continuous_update=False)
jump_button = widgets.Button(description='Study the jump', button_style='primary', icon='bolt')
jump_output = widgets.Output()

def draw_jump(_=None):
    with jump_output:
        clear_output(wait=True)
        f = FUNCTIONS[jump_f.value]
        c = jump_size.value
        d = jump_point.value
        alpha = lambda x: x + c * (x >= d).astype(float)
        naive, *_ = rs_sum(f, alpha, 0, 1, jump_n.value, 'midpoint')
        fine = np.linspace(0, 1, 40001)
        continuous_part = grid_integral(f(fine), fine)
        exact = continuous_part + c * float(f(np.array([d]))[0])
        jump_aware = exact
        ns = np.arange(2, max(25, jump_n.value + 1))
        errors = [abs(rs_sum(f, alpha, 0, 1, int(k), 'midpoint')[0] - exact) for k in ns]
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        x = np.linspace(0, 1, 2000)
        axes[0].plot(x, alpha(x), color=PURPLE)
        axes[0].axvline(d, color=RED, linestyle='--', label=f'jump at d={d:.2f}')
        axes[0].set_title(r'Integrator $\alpha(x)=x+cH_d(x)$')
        axes[0].set_xlabel('x')
        axes[0].legend()
        axes[1].semilogy(ns, np.maximum(errors, 1e-16), color=BLUE)
        axes[1].scatter([jump_n.value], [max(abs(naive-exact), 1e-16)], color=RED, zorder=4)
        axes[1].set_title('Error of a uniform midpoint sum')
        axes[1].set_xlabel('n')
        axes[1].set_ylabel('absolute error')
        plt.tight_layout()
        plt.show()
        display(HTML(
            f"<p><b>Uniform Stieltjes sum:</b> {naive:.10f}<br>"
            f"<b>Continuous contribution:</b> {continuous_part:.10f}<br>"
            f"<b>Atomic contribution cf(d):</b> {c*float(f(np.array([d]))[0]):.10f}<br>"
            f"<b>Jump-aware value:</b> {jump_aware:.10f}</p>"
        ))

jump_button.on_click(draw_jump)
display(widgets.VBox([jump_f, widgets.HBox([jump_size, jump_point]), jump_n, jump_button, jump_output]))
draw_jump()

---
## 5. Bounded variation and Jordan decomposition

For a function $\alpha:[a,b]\to\mathbb R$, the total variation is

$$
V_a^b(\alpha)=\sup_P\sum_{i=1}^n|\alpha(x_i)-\alpha(x_{i-1})|.
$$

A function has bounded variation when this supremum is finite. If $\alpha$ is continuously differentiable, then

$$V_a^b(\alpha)=\int_a^b|\alpha'(x)|\,dx.$$

Jordan's decomposition writes every bounded-variation function as the difference of two increasing functions. With $V(x)=V_a^x(\alpha)$ and $\alpha_0=\alpha(a)$, one convenient normalized choice is

$$
P(x)=\frac{V(x)+\alpha(x)-\alpha_0}{2},\qquad
N(x)=\frac{V(x)-\alpha(x)+\alpha_0}{2},
$$

so that $P,N$ are increasing and $\alpha(x)=\alpha_0+P(x)-N(x)$. The experiment uses $\alpha(x)=A\sin(kx)$ on $[0,2\pi]$, whose exact variation is $4A k$ when $A\ge0$ and $k$ is a positive integer.

In [ ]:
bv_amplitude = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='A:', continuous_update=False)
bv_frequency = widgets.IntSlider(value=2, min=1, max=8, step=1, description='k:', continuous_update=False)
bv_button = widgets.Button(description='Show decomposition', button_style='primary', icon='line-chart')
bv_output = widgets.Output()

def draw_bv(_=None):
    with bv_output:
        clear_output(wait=True)
        A = bv_amplitude.value
        k = bv_frequency.value
        x = np.linspace(0, 2 * np.pi, 12001)
        alpha = A * np.sin(k * x)
        derivative_abs = np.abs(A * k * np.cos(k * x))
        variation_path = cumulative_grid_integral(derivative_abs, x)
        P = 0.5 * (variation_path + alpha - alpha[0])
        N = 0.5 * (variation_path - alpha + alpha[0])
        numerical_variation = variation_path[-1]
        exact_variation = 4 * A * k
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(x, alpha, color=PURPLE, label='α(x)')
        axes[0].plot(x, variation_path, color=GREEN, label='V(0,x)')
        axes[0].set_title('Oscillation versus accumulated variation')
        axes[0].set_xlabel('x')
        axes[0].legend()
        axes[1].plot(x, P, color=BLUE, label='P(x), increasing')
        axes[1].plot(x, N, color=ORANGE, label='N(x), increasing')
        axes[1].plot(x, P - N + alpha[0], color=PURPLE, linestyle='--', label='P−N+α(0)')
        axes[1].set_title('Jordan decomposition')
        axes[1].set_xlabel('x')
        axes[1].legend()
        plt.tight_layout()
        plt.show()
        display(HTML(
            f"<p><b>Numerical total variation:</b> {numerical_variation:.9f}<br>"
            f"<b>Exact total variation 4Ak:</b> {exact_variation:.9f}<br>"
            f"<b>Absolute difference:</b> {abs(numerical_variation-exact_variation):.3e}</p>"
        ))

bv_button.on_click(draw_bv)
display(widgets.VBox([widgets.HBox([bv_amplitude, bv_frequency]), bv_button, bv_output]))
draw_bv()

---
## 6. Numerical computation and an a priori error bound

For continuous $f$, its modulus of continuity is

$$
\omega_f(\delta)=\sup\{|f(x)-f(y)|:x,y\in[a,b],\ |x-y|\le\delta\}.
$$

If $\alpha$ is increasing and $S(P,T;f,\alpha)$ is any tagged sum, then

$$
\left|S(P,T;f,\alpha)-\int_a^b f\,d\alpha\right|
\le \omega_f(\|P\|)\,[\alpha(b)-\alpha(a)].
$$

For $f(x)=\sin(\nu x)$, the Lipschitz estimate $|f(x)-f(y)|\le\nu|x-y|$ gives the simpler bound

$$
\text{error}\le \nu\|P\|[\alpha(b)-\alpha(a)].
$$

This bound is generally conservative: it is guaranteed by the theorem, but it need not predict the exact observed rate or exploit cancellation in the sum.

In [ ]:
error_frequency = widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5, description='ν:', continuous_update=False)
error_n = widgets.IntSlider(value=12, min=2, max=200, step=1, description='Intervals:', continuous_update=False)
error_button = widgets.Button(description='Compare error and bound', button_style='primary', icon='balance-scale')
error_output = widgets.Output()

def draw_error_bound(_=None):
    with error_output:
        clear_output(wait=True)
        nu = error_frequency.value
        f = lambda x: np.sin(nu * x)
        alpha = lambda x: x
        exact = (1 - np.cos(nu)) / nu
        ns = np.arange(2, max(40, error_n.value + 1))
        errors = np.array([abs(rs_sum(f, alpha, 0, 1, int(n), 'midpoint')[0] - exact) for n in ns])
        bounds = nu / ns
        selected_error = abs(rs_sum(f, alpha, 0, 1, error_n.value, 'midpoint')[0] - exact)
        selected_bound = nu / error_n.value
        
        fig, ax = plt.subplots()
        ax.loglog(ns, np.maximum(errors, 1e-18), color=BLUE, label='actual midpoint error')
        ax.loglog(ns, bounds, color=ORANGE, linestyle='--', label='Lipschitz/theorem bound ν/n')
        ax.scatter([error_n.value], [max(selected_error, 1e-18)], color=RED, zorder=5)
        ax.set_xlabel('number of subintervals n')
        ax.set_ylabel('error or bound')
        ax.set_title('Observed error versus guaranteed bound')
        ax.legend()
        plt.show()
        display(HTML(
            f"<p><b>Actual error at n={error_n.value}:</b> {selected_error:.3e}<br>"
            f"<b>Guaranteed bound:</b> {selected_bound:.3e}<br>"
            f"<b>Bound/error ratio:</b> {selected_bound/max(selected_error, 1e-18):.2f}</p>"
        ))

error_button.on_click(draw_error_bound)
display(widgets.VBox([widgets.HBox([error_frequency, error_n]), error_button, error_output]))
draw_error_bound()

---
## 7. The indefinite Riemann--Stieltjes integral

Fix $a$ and define

$$A(x)=\int_a^x f(t)\,d\alpha(t).$$

When $f$ is continuous and $\alpha\in C^1$, the reduction theorem yields

$$A(x)=\int_a^x f(t)\alpha'(t)\,dt,$$

and the ordinary Fundamental Theorem of Calculus gives

$$
\boxed{A'(x)=f(x)\alpha'(x)}.
$$

The plot below constructs the entire accumulated integral $A(x)$ and compares its numerical derivative with $f(x)\alpha'(x)$. Endpoint discrepancies in numerical differentiation are expected because finite differences are less symmetric there.

In [ ]:
indef_f = widgets.Dropdown(options=list(FUNCTIONS), value='f(x) = x²', description='Integrand:')
indef_alpha = widgets.Dropdown(options=list(INTEGRATORS), value='α(x) = exp(x)', description='Integrator:')
indef_x = widgets.FloatSlider(value=0.65, min=0.0, max=1.0, step=0.01, description='Evaluate x:', continuous_update=False)
indef_button = widgets.Button(description='Build A(x)', button_style='primary', icon='area-chart')
indef_output = widgets.Output()

def draw_indefinite(_=None):
    with indef_output:
        clear_output(wait=True)
        f = FUNCTIONS[indef_f.value]
        _, alpha_prime = INTEGRATORS[indef_alpha.value]
        x = np.linspace(0, 1, 12001)
        theoretical_derivative = f(x) * alpha_prime(x)
        A = cumulative_grid_integral(theoretical_derivative, x)
        numerical_derivative = np.gradient(A, x)
        j = int(round(indef_x.value * (len(x) - 1)))
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(x, A, color=BLUE, label='A(x)')
        axes[0].scatter([x[j]], [A[j]], color=RED, zorder=5)
        axes[0].set_title('Accumulated Stieltjes integral')
        axes[0].set_xlabel('x')
        axes[0].legend()
        axes[1].plot(x, numerical_derivative, color=ORANGE, label="numerical A'(x)")
        axes[1].plot(x, theoretical_derivative, color=GREEN, linestyle='--', label="f(x)α'(x)")
        axes[1].set_title('Fundamental derivative identity')
        axes[1].set_xlabel('x')
        axes[1].legend()
        plt.tight_layout()
        plt.show()
        display(HTML(
            f"<p><b>A({x[j]:.2f}) ≈</b> {A[j]:.10f}<br>"
            f"<b>Numerical A′({x[j]:.2f}) ≈</b> {numerical_derivative[j]:.10f}<br>"
            f"<b>f(x)α′(x) =</b> {theoretical_derivative[j]:.10f}</p>"
        ))

indef_button.on_click(draw_indefinite)
display(widgets.VBox([widgets.HBox([indef_f, indef_alpha]), indef_x, indef_button, indef_output]))
draw_indefinite()

---
## 8. Improper Riemann--Stieltjes integrals

An integral on an unbounded interval is defined through a limit:

$$
\int_0^\infty f(x)\,d\alpha(x)=\lim_{R\to\infty}\int_0^R f(x)\,d\alpha(x),
$$

provided that this limit exists. Consider $f(x)=e^{-x}$ and $\alpha(x)=x^2$. Since $d\alpha=2x\,dx$,

$$
\int_0^\infty e^{-x}\,d(x^2)=\int_0^\infty 2xe^{-x}\,dx=2.
$$

For finite $R$, direct integration gives

$$I(R)=2-2(R+1)e^{-R}.$$

Move the truncation point to observe the improper limit and the remaining tail $2-I(R)$.

In [ ]:
improper_R = widgets.FloatSlider(value=4.0, min=0.25, max=12.0, step=0.25, description='R:', continuous_update=False)
improper_button = widgets.Button(description='Show truncation', button_style='primary', icon='long-arrow-right')
improper_output = widgets.Output()

def draw_improper(_=None):
    with improper_output:
        clear_output(wait=True)
        R = improper_R.value
        r = np.linspace(0, 12, 1200)
        I = 2 - 2 * (r + 1) * np.exp(-r)
        selected = 2 - 2 * (R + 1) * np.exp(-R)
        tail = 2 - selected
        fig, ax = plt.subplots()
        ax.plot(r, I, color=BLUE, label=r'$I(R)$')
        ax.axhline(2, color=GREEN, linestyle='--', label='improper limit = 2')
        ax.scatter([R], [selected], color=RED, s=60, zorder=5)
        ax.vlines(R, selected, 2, color=RED, linestyle=':', label='remaining tail')
        ax.set_xlabel('truncation point R')
        ax.set_ylabel('truncated integral')
        ax.set_title('Convergence of an improper Stieltjes integral')
        ax.legend()
        plt.show()
        display(HTML(
            f"<p><b>I({R:.2f}) =</b> {selected:.10f}<br>"
            f"<b>Tail 2 − I(R) =</b> {tail:.3e}</p>"
        ))

improper_button.on_click(draw_improper)
display(widgets.VBox([improper_R, improper_button, improper_output]))
draw_improper()

---
## 9. Probability: expectation as a Stieltjes integral

If $F_X$ is the cumulative distribution function of a random variable $X$, then

$$\mathbb E[g(X)]=\int_{-\infty}^{\infty}g(x)\,dF_X(x),$$

whenever the integral is absolutely convergent. This single formula includes:

$$
\mathbb E[g(X)]=\sum_j g(x_j)p_j
\quad\text{(discrete case)},
$$

and

$$
\mathbb E[g(X)]=\int_{-\infty}^{\infty}g(x)p_X(x)\,dx
\quad\text{(absolutely continuous case)}.
$$

Now let $X$ be uniform on $[0,1]$ with probability $p$ and equal to $x_0$ with probability $1-p$. Then

$$
F_X(x)=pF_U(x)+(1-p)H_{x_0}(x)
$$

and linearity in the integrator gives

$$
\boxed{\mathbb E[g(X)]=p\int_0^1g(x)\,dx+(1-p)g(x_0)}.
$$

In [ ]:
prob_p = widgets.FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05, description='Uniform p:', continuous_update=False)
prob_x0 = widgets.FloatSlider(value=0.75, min=0.0, max=1.0, step=0.05, description='Atom x₀:', continuous_update=False)
prob_g = widgets.Dropdown(options=['g(x) = x', 'g(x) = x²', 'g(x) = exp(x)'], value='g(x) = x', description='Function:')
prob_n = widgets.IntSlider(value=40, min=5, max=300, step=5, description='Intervals:', continuous_update=False)
prob_button = widgets.Button(description='Compute expectation', button_style='primary', icon='bar-chart')
prob_output = widgets.Output()

def draw_probability(_=None):
    with prob_output:
        clear_output(wait=True)
        p = prob_p.value
        x0 = prob_x0.value
        g_map = {
            'g(x) = x': (lambda x: x, 0.5),
            'g(x) = x²': (lambda x: x**2, 1/3),
            'g(x) = exp(x)': (np.exp, np.e - 1),
        }
        g, uniform_expectation = g_map[prob_g.value]
        exact = p * uniform_expectation + (1 - p) * float(g(np.array([x0]))[0])
        exact_mean = p / 2 + (1 - p) * x0
        exact_second = p / 3 + (1 - p) * x0**2
        exact_variance = exact_second - exact_mean**2
        
        def uniform_cdf(x):
            return np.clip(x, 0, 1)
        def mixed_cdf(x):
            return p * uniform_cdf(x) + (1 - p) * (x >= x0).astype(float)
        
        numerical, *_ = rs_sum(g, mixed_cdf, -0.1, 1.1, prob_n.value, 'midpoint')
        x = np.linspace(-0.1, 1.1, 2401)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(x, mixed_cdf(x), color=PURPLE, label='mixed CDF')
        axes[0].axvline(x0, color=RED, linestyle='--', label='atom')
        axes[0].set_ylim(-0.04, 1.04)
        axes[0].set_xlabel('x')
        axes[0].set_ylabel(r'$F_X(x)$')
        axes[0].set_title('Continuous growth plus a jump')
        axes[0].legend()
        labels = ['uniform part', 'atomic part', 'total']
        values = [p * uniform_expectation, (1 - p) * float(g(np.array([x0]))[0]), exact]
        axes[1].bar(labels, values, color=[BLUE, ORANGE, GREEN], alpha=0.82)
        axes[1].set_title(r'Contributions to $\mathbb{E}[g(X)]$')
        axes[1].tick_params(axis='x', rotation=12)
        plt.tight_layout()
        plt.show()
        display(HTML(
            f"<p><b>Exact E[g(X)]:</b> {exact:.10f}<br>"
            f"<b>Direct Stieltjes-sum approximation:</b> {numerical:.10f}<br>"
            f"<b>Mean E[X]:</b> {exact_mean:.10f}<br>"
            f"<b>Variance Var(X):</b> {exact_variance:.10f}</p>"
        ))

prob_button.on_click(draw_probability)
display(widgets.VBox([widgets.HBox([prob_p, prob_x0]), widgets.HBox([prob_g, prob_n]), prob_button, prob_output]))
draw_probability()

---
## 10. Concept check

Use the quiz as a diagnostic tool. After choosing an answer, press **Check answer**. The feedback explains the mathematical reason; it does more than report whether the choice is correct. Use **Next question** to continue.

In [ ]:
QUIZ = [
    {
        'question': '1. If α(x)=x, what does the Riemann–Stieltjes integral become?',
        'options': ['A weighted series', 'The ordinary Riemann integral', 'Always zero'],
        'answer': 'The ordinary Riemann integral',
        'explanation': 'Because α(x_i)−α(x_{i−1})=x_i−x_{i−1}, the defining sums are ordinary Riemann sums.'
    },
    {
        'question': '2. What does a jump of size c in α at d contribute when f is continuous?',
        'options': ['c f(d)', 'c α(d)', 'Nothing'],
        'answer': 'c f(d)',
        'explanation': 'The full increment c is concentrated at d, so the limiting tagged contribution is c f(d).'
    },
    {
        'question': '3. If α is continuously differentiable, which reduction is correct?',
        'options': ['∫f dα = ∫fα dx', '∫f dα = ∫fα′ dx', '∫f dα = ∫f′α′ dx'],
        'answer': '∫f dα = ∫fα′ dx',
        'explanation': 'Locally, Δα is approximately α′(x)Δx; therefore α′ is the ordinary integration density.'
    },
    {
        'question': '4. What does U(P,f,α)−L(P,f,α) measure?',
        'options': ['Only the variation of α', 'A weighted sum of local oscillations of f', 'The integral itself'],
        'answer': 'A weighted sum of local oscillations of f',
        'explanation': 'The exact identity is U−L=Σ osc(f;I_i) Δα_i when α is increasing.'
    },
    {
        'question': '5. Why can a mixed expectation contain both an integral and a sum?',
        'options': ['Because a CDF may have continuous growth and jumps', 'Because expectation is not linear', 'Only because of numerical error'],
        'answer': 'Because a CDF may have continuous growth and jumps',
        'explanation': 'The continuous part of the CDF generates an ordinary integral; each jump generates an atomic weighted value.'
    },
]

quiz_state = {'index': 0, 'score': 0, 'answered': set()}
quiz_title = widgets.HTML()
quiz_choice = widgets.RadioButtons(options=[])
quiz_check = widgets.Button(description='Check answer', button_style='success', icon='check')
quiz_next = widgets.Button(description='Next question', button_style='primary', icon='arrow-right')
quiz_restart = widgets.Button(description='Restart quiz', icon='refresh')
quiz_feedback = widgets.Output()

def load_quiz_question():
    item = QUIZ[quiz_state['index']]
    quiz_title.value = f"<h4>{item['question']}</h4><p>Question {quiz_state['index']+1} of {len(QUIZ)}</p>"
    quiz_choice.options = item['options']
    quiz_choice.value = None
    with quiz_feedback:
        clear_output()

def check_quiz_answer(_):
    with quiz_feedback:
        clear_output(wait=True)
        if quiz_choice.value is None:
            display(HTML("<p style='color:#b45309'><b>Please select an answer first.</b></p>"))
            return
        item = QUIZ[quiz_state['index']]
        correct = quiz_choice.value == item['answer']
        if quiz_state['index'] not in quiz_state['answered']:
            quiz_state['answered'].add(quiz_state['index'])
            quiz_state['score'] += int(correct)
        color = '#166534' if correct else '#b91c1c'
        heading = 'Correct.' if correct else f"Not quite. The correct answer is: {item['answer']}."
        display(HTML(
            f"<div style='border-left:4px solid {color};padding:8px 12px'>"
            f"<b style='color:{color}'>{heading}</b><br>{item['explanation']}"
            f"<br><small>Current score: {quiz_state['score']}/{len(quiz_state['answered'])}</small></div>"
        ))

def next_quiz_question(_):
    quiz_state['index'] = (quiz_state['index'] + 1) % len(QUIZ)
    load_quiz_question()

def restart_quiz(_):
    quiz_state.update(index=0, score=0, answered=set())
    load_quiz_question()

quiz_check.on_click(check_quiz_answer)
quiz_next.on_click(next_quiz_question)
quiz_restart.on_click(restart_quiz)
display(widgets.VBox([
    quiz_title, quiz_choice,
    widgets.HBox([quiz_check, quiz_next, quiz_restart]),
    quiz_feedback
]))
load_quiz_question()

---
## 11. Guided investigations

Use the notebook as a laboratory for the following problems. Record both a conjecture from the graph and a mathematical justification.

### Investigation A — the role of the integrator

Fix $f(x)=x^2$ and compare $\alpha(x)=x$, $x^2$, and $e^x$ on $[0,1]$. Before computing, predict which integrator places the greatest weight near $x=1$. Then explain the results through $\alpha'(x)$.

### Investigation B — refinement and oscillation

For $f(x)=\sin(2\pi x)$, make a table of $U(P_n)-L(P_n)$ for $n=4,8,16,32,64$. Estimate the experimental order of decay and compare it with the uniform-continuity argument.

### Investigation C — jump-aware quadrature

Let $\alpha(x)=x+2H_{0.37}(x)$ and $f(x)=e^x$. Compare the uniform midpoint sum with

$$\int_0^1 e^x\,dx+2e^{0.37}.$$

Explain why explicitly separating the jump is exact for the atomic part.

### Investigation D — probability mixture

Derive, before using the controls,

$$
\mathbb E[X]=\frac p2+(1-p)x_0,\qquad
\operatorname{Var}(X)=\frac p3+(1-p)x_0^2-\left(\frac p2+(1-p)x_0\right)^2.
$$

Check several limiting cases, including $p=0$ and $p=1$.

## Final perspective

The same increment-based definition explains all of the phenomena explored here:

- $\alpha(x)=x$ produces ordinary length;
- a differentiable $\alpha$ produces the density $\alpha'(x)$;
- a jump of $\alpha$ produces a point mass;
- bounded variation permits positive and negative accumulated motion through Jordan decomposition; and
- a cumulative distribution function turns Stieltjes integration into expectation.

In compact form,

$$
\text{lengths, densities, atoms, and probability weights}
\quad\longleftrightarrow\quad
\alpha(x_i)-\alpha(x_{i-1}).
$$

That is the unifying idea of the Riemann--Stieltjes integral.